In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains

import time
import pandas as pd
from pathlib import Path

In [2]:
DOMAIN_URL = "https://d2.naver.com/"

SAVE_DIR = Path() / "datas"
if not (SAVE_DIR).exists():
  SAVE_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FILE = SAVE_DIR / "naver.xlsx"

TEST = True

In [3]:
# 확인하는 페이지 개수
TEST_COUNT = 5

options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options)

wait = WebDriverWait(driver, 5)


driver.get(DOMAIN_URL)

# list[df[title, link, summary, year, month, date, view_count]]
pages_df_list = []

go_on_flag = True
i = 1
# 페이지 순회하며 크롤링 시작
while go_on_flag:
  # 테스트면 테스트만큼 해주기
  if TEST:
    if TEST_COUNT <= 0:
      break
    TEST_COUNT -= 1

  print(f"\n --- {i} --- \n")
  i += 1
  print(driver.current_url)

  # 렌더링 약간 기다리기 (+너무 빨리 진행을 막기 위해)
  time.sleep(1)

  # 게시되어있는 모든 게시글들 읽기
  posts = wait.until(
    EC.presence_of_all_elements_located(
      (By.CSS_SELECTOR, "div.contents div.post_article > div.cont_post")
    )
  )

  # 현재 페이지의 포스팅 모두 긁기
  post_meta = []
  for p in posts:
    # print("\n --- --- \n")
    title = p.find_element(By.CSS_SELECTOR, "h2").text
    link = str(p.find_element(By.CSS_SELECTOR, "a").get_attribute("href"))
    summary = p.find_element(By.CSS_SELECTOR, "div.post_txt").text
    # ['2026.06.15', '|', '9181']
    [post_date, view_count] = list(map(lambda x: x.text, p.find_elements(By.CSS_SELECTOR, "dd")))[0::2]
    post_date = post_date.split(".")



    # print(title)
    # print(link)
    # print(summary)
    # print(post_date[0])
    # print(post_date[1])
    # print(post_date[2])
    # print(view_count)

    title = title
    link = link
    summary = summary
    year  = post_date[0]
    month = post_date[1]
    date = post_date[2]
    view_count = view_count

    data = {
      "title": title,
      "link": link,
      "summary": summary,
      "year": year,
      "month": month,
      "date": date,
      "view_count": view_count
    }

    post_meta.append(data)

  # for문 끝나고 현재 만든 메타 df 리스트에 넣어주기
  post_meta_df = pd.DataFrame(post_meta, columns=list(post_meta[0].keys()))
  pages_df_list.append(post_meta_df)


  # 현재 페이지가 마지막 페이지네이션 버튼이 아니면 다음 페이지 가주기
  page_buttons = wait.until(
    EC.presence_of_all_elements_located(
      (By.CSS_SELECTOR, "div.paginate_box > div.paginate a")
    )
  )

  print(list(map(lambda x: x.text, page_buttons)))

  click_next = False
  for pb in page_buttons:
    pagenate_button_classes = str(pb.get_attribute("class")).split(" ")
    # print(pagenate_button_classes)
    # print(pb.text)

    # 현재 페이지가 있는 버튼 찾으면 플래그 키기
    if "btn_num" in pagenate_button_classes and "select" in pagenate_button_classes:
      click_next = True
      continue
    # 다음 버튼 누르기가 있으며 현재 선택된 버튼이 아니면 눌러주기
    if click_next and "btn_num" in pagenate_button_classes and "select" not in pagenate_button_classes:
      print(f"${pb.text} 누름!")
      pb.click()
      break

  # 만약 다음 버튼 누르려 했는데 누르지 않고 그냥 
  if page_buttons[-2].get_attribute("class") == "btn_num select":
    break

  print("여기까지 옴")

with pd.ExcelWriter(SAVE_FILE) as writer:
  for index, page_meta in enumerate(pages_df_list):
    page_meta.to_excel(writer, sheet_name=f"page_{index+1}", index=False)

  







 --- 1 --- 

https://d2.naver.com/home
['1', '2', '3', '4', '5', '다음']
$2 누름!
여기까지 옴

 --- 2 --- 

https://d2.naver.com/home?page=1
['1', '2', '3', '4', '5', '다음']
$3 누름!
여기까지 옴

 --- 3 --- 

https://d2.naver.com/home?page=2
['1', '2', '3', '4', '5', '다음']
$4 누름!
여기까지 옴

 --- 4 --- 

https://d2.naver.com/home?page=3
['이전', '2', '3', '4', '5', '6', '다음']
$5 누름!
여기까지 옴

 --- 5 --- 

https://d2.naver.com/home?page=4
['이전', '3', '4', '5', '6', '7', '다음']
$6 누름!
여기까지 옴


In [10]:
var = ['2026.06.15', '|', '9181']

[post_date, view] = var[0::2]
print(post_date, view)

2026.06.15 9181


In [13]:
var = {
  "asdf": "asdf",
  "asdf": "aaaa"
}

list(var.keys())

['asdf']